In [ ]:
"""
Your FIRST test file.

How pytest works, in one breath:
  - A test is a function whose name starts with `test_`.
  - Inside it you DO something, then `assert <thing that must be true>`.
  - If every assert is true -> the test PASSES. If one is false (or the code
    crashes) -> the test FAILS and pytest shows you exactly where.

We test the SIMPLEST part of the system: `simulate_silo_response`. It takes a
machine ("plant") + fixed controller knobs ("gains") and returns how the machine
behaved (a "success" flag + some performance "metrics"). No fake-AI, no server,
runs in a fraction of a second. This is the same function the /silo/simulate
web endpoint calls under the hood.
"""

# `pytest` gives us nice helpers (used later for the parametrize example).
import pytest

# This is the function under test. It lives in api/silo_service.py.
from api.silo_service import simulate_silo_response

# ---------------------------------------------------------------------------
# Test 1: the "happy path" — everything normal, expect success.
# ---------------------------------------------------------------------------
def test_ball_beam_simulation_succeeds():
    # ARRANGE: describe what we want to simulate.
    # This dict is exactly the JSON body you saw in the /docs "Try it out" box.
    request = {
        "system_name": "ball_beam",           # which built-in machine
        "controller_type": "PID",             # the kind of controller
        "gains": {"Kp": 5.0, "Ki": 0.1, "Kd": 1.0},  # the tuning knobs
        "dt": 0.01,                           # simulation time step (seconds)
        "max_time": 5.0,                      # how long to simulate (seconds)
    }

    # ACT: run the simulation.
    result = simulate_silo_response(request)

    # ASSERT: check the result is shaped the way we expect.
    # `result` is a dict. Let's look at what must be true.
    assert result["success"] is True, f"Simulation should succeed, got: {result.get('message')}"

    # There must be a "metrics" dict with actual numbers in it.
    assert "metrics" in result
    assert result["metrics"], "metrics dict should not be empty"

    # mse = mean squared error (how far the machine was from the goal, on average).
    # It must exist and be a real, non-negative number.
    mse = result["metrics"]["mse"]
    assert mse >= 0, f"mse should never be negative, got {mse}"

    # The trajectory is the machine's position over time — it should have points.
    assert len(result["trajectory"]) > 0


# ---------------------------------------------------------------------------
# Test 2: a "failure mode" — a machine name that doesn't exist.
# The system should NOT crash; it should fail gracefully.
# (Testing that bad input is handled cleanly is just as important as the happy path.)
# ---------------------------------------------------------------------------
def test_unknown_system_does_not_crash():
    request = {
        "system_name": "this_machine_does_not_exist",
        "gains": {"Kp": 1.0},
    }

    # This must NOT raise an exception — the service catches errors and returns a dict.
    result = simulate_silo_response(request)

    # We simply assert it returned a dict with a success flag we can read.
    assert isinstance(result, dict)
    assert "success" in result
    # NOTE: We deliberately do not assert success is False here, because the
    # provided code falls back to the ball_beam plant for unknown names
    # (see create_system in src/systems.py). Documenting that behaviour IS the
    # point of an edge-case test. We'll tighten this once we decide the desired
    # behaviour — for now we just prove it stays alive and well-formed.


# ---------------------------------------------------------------------------
# Test 3: PARAMETRIZE — run the SAME test for MANY machines automatically.
# Instead of copy-pasting Test 1 three times, pytest runs it once per value.
# You'll see it reported as 3 separate tests: [ball_beam], [dc_motor], [inverted_pendulum].
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("system_name", ["ball_beam", "dc_motor", "inverted_pendulum"])
def test_all_builtin_plants_simulate(system_name):
    request = {
        "system_name": system_name,
        "controller_type": "PID",
        "gains": {"Kp": 5.0, "Ki": 0.1, "Kd": 1.0},
        "dt": 0.01,
        "max_time": 5.0,
    }

    result = simulate_silo_response(request)

    assert result["success"] is True, (
        f"{system_name} failed: {result.get('message')}"
    )
    assert result["metrics"]["mse"] >= 0


ModuleNotFoundError: No module named 'api'

: 